# CAMIIRA v0.3.0: end-to-end camera tutorial

This notebook installs the tagged **CAMIIRA v0.3.0** release, downloads a small public example dataset, converts paired BMP camera images, calculates calibration parameters, detects colonies, and validates the generated image, CSV, and COCO JSON outputs.

> The three example plates make this tutorial quick and reproducible. For production calibration, use a larger representative reference panel collected with your own camera setup.

## 1. Install the release

The normal Colab path installs directly from the immutable Git tag. The local-source branch is only used by CAMIIRA maintainers to validate this notebook before publishing the tag.

In [ ]:
import os
import subprocess
import sys

RELEASE = "v0.3.0"
REPOSITORY = "https://github.com/whatever60/camiira.git"

if os.environ.get("CAMIIRA_USE_LOCAL_SOURCE") == "1":
    local_source = os.environ["CAMIIRA_LOCAL_SRC"]
    sys.path.insert(0, local_source)
    os.environ["PYTHONPATH"] = os.pathsep.join(
        filter(None, [local_source, os.environ.get("PYTHONPATH")])
    )
    print("Maintainer validation: using local CAMIIRA source.")
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"camiira @ git+{REPOSITORY}@{RELEASE}",
        ],
        check=True,
    )

import camiira

print(f"CAMIIRA {camiira.__version__} is ready.")

## 2. Download and verify the example camera data

Each plate has exactly two BMP files. Sorted by filename, the first is the red-light image and the second is the white-light image. The plate barcode is the text before the first underscore.

In [ ]:
import hashlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA_URL = (
    "https://easy-amplicon-camii-test-data.s3.amazonaws.com/"
    "camiira/v0.2.0/camiira-example-camera-v0.2.0.zip"
)
DATA_SHA256 = "73aa884e0707799b458a4cfc397d0ae0e315cc6943f0673a6a88a4a891f83a19"
BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORK_DIR = BASE_DIR / "camiira-v0.3.0-tutorial"
ARCHIVE_PATH = BASE_DIR / "camiira-example-camera-v0.2.0.zip"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
actual_sha256 = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
assert actual_sha256 == DATA_SHA256, (actual_sha256, DATA_SHA256)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(WORK_DIR)

DATA_DIR = WORK_DIR / "camiira-example-v0.2.0"
BMP_DIR = DATA_DIR / "bmp"
SMALL_CONFIG_PATH = WORK_DIR / "configure_small.yaml"

if os.environ.get("CAMIIRA_USE_LOCAL_SOURCE") == "1":
    local_repo = Path(os.environ["CAMIIRA_LOCAL_SRC"]).resolve().parent
    shutil.copy2(
        local_repo / "test_data" / "configs" / "configure_small.yaml",
        SMALL_CONFIG_PATH,
    )
else:
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/whatever60/camiira/{RELEASE}/"
        "test_data/configs/configure_small.yaml",
        SMALL_CONFIG_PATH,
    )
TRANSFORMED_DIR = WORK_DIR / "transformed"
CALIBRATION_PATH = WORK_DIR / "calibration.npz"
RESULTS_DIR = WORK_DIR / "results"

bmp_files = sorted(BMP_DIR.glob("*.bmp"))
assert len(bmp_files) == 6
print(f"Verified and extracted {len(bmp_files)} BMP files to {DATA_DIR}")

## 3. Transform the BMP pairs

This creates `red_rgb`, `red_grayscale`, `white_rgb`, and `white_grayscale`. Calibration and batch detection consume the matching `red_rgb` and `white_rgb` images. The tutorial intentionally leaves filenames unsuffixed, which exercises support for unsuffixed single-time-point images.

In [ ]:
def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout.strip())


run_module(
    "camiira.data_transform",
    "process_bmp",
    "--input_dir",
    BMP_DIR,
    "--output_dir",
    TRANSFORMED_DIR,
)

for subdirectory in ("red_rgb", "red_grayscale", "white_rgb", "white_grayscale"):
    files = sorted((TRANSFORMED_DIR / subdirectory).glob("*.png"))
    assert len(files) == 3, (subdirectory, files)
print("BMP transformation produced all 12 expected PNG files.")

## 4. Calculate calibration parameters

The calibration step estimates non-uniform illumination from the transformed reference images. The same `configure_small.yaml` RGB-camera configuration is used here and for colony detection.

In [ ]:
run_module(
    "camiira.calc_calib_params",
    "--input_dir",
    TRANSFORMED_DIR,
    "--output_path",
    CALIBRATION_PATH,
    "--config_path",
    SMALL_CONFIG_PATH,
)
assert CALIBRATION_PATH.is_file()
print(f"Calibration parameters written to {CALIBRATION_PATH}")

## 5. Detect colonies

`configure_small.yaml` is the RGB-camera configuration. No `--time_label` is needed here: the default `max` selector correctly chooses unsuffixed images when they are the only images for a plate.

In [ ]:
run_module(
    "camiira.detect_colonies",
    "--input_path",
    TRANSFORMED_DIR,
    "--output_dir",
    RESULTS_DIR,
    "--calib_param_path",
    CALIBRATION_PATH,
    "--config_path",
    SMALL_CONFIG_PATH,
)
print(f"Detection outputs written to {RESULTS_DIR}")

## 6. Validate tables and annotations

These checks verify that every example plate produced a non-empty metadata table, that every table row has a corresponding COCO annotation, and that key numeric measurements are finite and within the image boundaries. Counts may vary slightly across compatible dependency versions, so the notebook checks invariants rather than hard-coding one colony count.

In [ ]:
import json

import numpy as np
import pandas as pd
from PIL import Image

EXPECTED_BARCODES = ["FT1AR80", "FT1AR81", "FT4AR80"]
numeric_columns = [
    "area",
    "center_x",
    "center_y",
    "circularity",
    "convexity",
    "radius",
    "Graymean",
    "Bepimean",
    "Gepimean",
    "Repimean",
]
summaries = []

with np.load(CALIBRATION_PATH) as calibration:
    assert calibration["image_trans_calib"].shape == (1000, 1600)
    assert calibration["image_epi_calib"].shape == (1000, 1600, 3)
    assert all(np.isfinite(calibration[name]).all() for name in calibration.files)

for barcode in EXPECTED_BARCODES:
    csv_path = RESULTS_DIR / f"{barcode}_metadata.csv"
    json_path = RESULTS_DIR / f"{barcode}_annot.json"
    red_overlay = RESULTS_DIR / f"{barcode}_gs_red_contour.jpg"
    white_overlay = RESULTS_DIR / f"{barcode}_rgb_white_contour.jpg"
    assert all(path.is_file() for path in (csv_path, json_path, red_overlay, white_overlay))

    metadata = pd.read_csv(csv_path)
    annotations = json.loads(json_path.read_text())
    width, height = Image.open(TRANSFORMED_DIR / "red_grayscale" / f"{barcode}.png").size

    assert len(metadata) > 0
    assert metadata["plate_barcode"].eq(barcode).all()
    assert np.isfinite(metadata[numeric_columns].to_numpy()).all()
    assert (metadata[["area", "radius"]] > 0).all().all()
    assert metadata["center_x"].between(0, width).all()
    assert metadata["center_y"].between(0, height).all()
    assert len(annotations["images"]) == 1
    assert len(annotations["annotations"]) == len(metadata)

    summaries.append(
        {
            "plate": barcode,
            "colonies": len(metadata),
            "median_area_px": round(metadata["area"].median(), 1),
            "median_radius_px": round(metadata["radius"].median(), 1),
            "mean_circularity": round(metadata["circularity"].mean(), 3),
        }
    )

summary = pd.DataFrame(summaries)
display(summary)
print("All calibration, CSV, COCO JSON, and coordinate checks passed.")

## 7. Inspect the colony overlays

Black outlines show detected colony boundaries and green marks show their centers. The red-light grayscale view drives detection; the white-light RGB view provides a paired visual cross-check. Look for outlines that follow visible colonies rather than plate edges, glare, or broad background regions.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(len(EXPECTED_BARCODES), 2, figsize=(14, 11))
for row, barcode in enumerate(EXPECTED_BARCODES):
    views = [
        (RESULTS_DIR / f"{barcode}_gs_red_contour.jpg", "red grayscale", "gray"),
        (RESULTS_DIR / f"{barcode}_rgb_white_contour.jpg", "white RGB", None),
    ]
    for column, (path, label, color_map) in enumerate(views):
        axes[row, column].imshow(Image.open(path), cmap=color_map)
        axes[row, column].set_title(f"{barcode}: {label}")
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

## Result

A successful run ends with the validation message above and six overlay images whose contours align with visible colonies. The complete generated dataset is in `camiira-v0.3.0-tutorial/results`. These automated and visual checks are pipeline sanity checks, not biological ground-truth validation.